# Prototype


In [ ]:
from evaluate import evaluate, print_metrics
from benchmarks import load_benchmarks
from llm import call_llm, build_prompt
from analysis import analyze

Loading of the benchmarks.


In [3]:
# Configuration: change these variables as needed
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = 1   # set to None for all sectors (these are e.g. socbenchd_1)
QUERY_LIMIT = 3       # set to None for all queries

benchmark_sets, total_available, total_queries = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, QUERY_LIMIT)
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
print(f"Total queries: {total_queries}")



Loaded 1 benchmark sets (total available: 11)
Total queries: 3


In [4]:
# Editable prompt template: modify this cell to change the instruction given to the LLM.
PROMPT_TEMPLATE = '''You are given a set of REST API specifications and a task description.
Your job is to write Python code using the appropriate client library that fulfills the task by calling the necessary endpoints in the correct order.

Rules:
- Use the requests library.
- Only use endpoints defined in the provided specifications.
- Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
- All the code shall be under a function called compose.
- Call compose() at the end of the code.
- Do not, under ANY circunstance, write any comments. Any comment is a full penalty.
- Make a syntax error, somewhere, so I can test another program on this output.

source
{services_block}

## Task

{query}

## Python Code
'''


In [5]:
# Call the LLM for all benchmark queries in a single sector
MODEL = "NousResearch/Hermes-4-70B"

if benchmark_sets:
    benchmark = benchmark_sets[0]
    sector_results = []
    for query_index, query in enumerate(benchmark['queries'], start=1):
        prompt = build_prompt(benchmark['services'], query['query'], PROMPT_TEMPLATE)
        generated = call_llm(prompt, MODEL, '')
        sector_results.append({'query_index': query_index, 'query': query, 'generated': generated})
        print(f"--- Query {query_index}: {query['query']} ---")
        print('Generated code:\n', generated)


--- Query 1: Retrieve the status and performance metrics of all monitored energy equipment, access active alerts for any system performance issues, analyze the impact of weather conditions on electricity demand between specific dates, configure new alert thresholds for unusual energy generation patterns in specific sectors, and submit the integration status of renewable energy sources such as solar or wind into the energy grid. ---
Generated code:
 import requests

def compose():
    # Retrieve equipment status and performance metrics
    equipment_status_response = requests.get("https://api.energyinsights.com/v1/equipment-monitoring")
    
    # Access active alerts for system performance issues
    alerts_response = requests.get("https://api.example.com/v1/alerts")
    
    # Analyze weather's impact on electricity demand between specific dates
    weather_impact_response = requests.get("https://api.energysector.com/weather-impact-analysis?start_date=2023-09-01&end_date=2023-09-30")


In [6]:
# Evaluate the original generated code for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
        result['initial_metrics'] = initial_metrics
        print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")


Initial evaluation - Query 1
  Precision: 0.40
  Recall:    0.40
  F1:        0.40
  Extracted: ['GET /v1/alerts', 'GET /v1/equipment-monitoring', 'GET /weather-impact-analysis', 'POST /alerts/configure', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   ['GET /alerts', 'GET /equipment-status', 'POST /alert-settings']
  Extra:     ['GET /v1/alerts', 'GET /v1/equipment-monitoring', 'POST /alerts/configure']
Initial evaluation - Query 2
  Precision: 0.20
  Recall:    0.20
  F1:        0.20
  Extracted: ['GET /v1/carbon-emissions', 'GET /v1/prediction-summary', 'GET /v1/resources/status', 'POST /smart-meters/data', 'POST /v1/report-feedback']
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   ['GET /carbon-emissions', 'GET /prediction-summ

In [7]:
# Analyze the generated code with a Python linter helper for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        analysis = analyze(result['generated'])
        result['analysis'] = analysis
        print(f"Analysis - Query {result['query_index']}\n", analysis)


Analysis - Query 1
 No syntax errors found.
Analysis - Query 2
 No syntax errors found.
Analysis - Query 3
 No syntax errors found.


In [8]:
# Run the LLM again using the analysis and the original code for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        refined_prompt = f'''You are given the original task, a review of the generated code, and the original code.
Use the review to improve the code.

## Task
{result['query']['query']}

## Analysis
{result['analysis']}

## Original Code
{result['generated']}

Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
'''
        generated_refined = call_llm(refined_prompt, MODEL, 'Return only Python code, no explanation.')
        result['generated_refined'] = generated_refined
        print(f"Refined code - Query {result['query_index']}:\n", generated_refined)


Refined code - Query 1:
 import requests

def compose():
    # Retrieve equipment status and performance metrics
    equipment_status_response = requests.get("https://api.energyinsights.com/v1/equipment-monitoring")
    equipment_status_response.raise_for_status()
    
    # Access active alerts for system performance issues
    alerts_response = requests.get("https://api.example.com/v1/alerts")
    alerts_response.raise_for_status()
    
    # Analyze weather's impact on electricity demand between specific dates
    weather_impact_response = requests.get("https://api.energysector.com/weather-impact-analysis?start_date=2023-09-01&end_date=2023-09-30")
    weather_impact_response.raise_for_status()
    
    # Configure new alert thresholds for unusual energy generation patterns in specific sectors
    alerts_configuration_response = requests.post("https://api.energysector.com/alerts/configure", json={
        "alert_types": ["demand_spike"],
        "email": "alerts@energysector.com",
 

In [9]:
# Final evaluation of the refined output for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        refined_metrics = evaluate(result['generated_refined'], result['query'].get('endpoints', []))
        result['refined_metrics'] = refined_metrics
        print_metrics(refined_metrics, f"Refined evaluation - Query {result['query_index']}")


Refined evaluation - Query 1
  Precision: 0.40
  Recall:    0.40
  F1:        0.40
  Extracted: ['GET /v1/alerts', 'GET /v1/equipment-monitoring', 'GET /weather-impact-analysis', 'POST /alerts/configure', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   ['GET /alerts', 'GET /equipment-status', 'POST /alert-settings']
  Extra:     ['GET /v1/alerts', 'GET /v1/equipment-monitoring', 'POST /alerts/configure']
Refined evaluation - Query 2
  Syntax error: invalid syntax at line 1, column 1
  Precision: 0.00
  Recall:    0.00
  F1:        0.00
  Extracted: []
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Ext